# vLLM_Batch — Test Runner

Enqueues TC1–TC4 to the Delta queue, triggers the vLLM_Batch job, polls for completion, and writes performance metrics to the shared perf table.

**This notebook does NOT run inference** — it orchestrates and measures the batch job.

### Cluster Requirements

| Setting | Value |
|---------|-------|
| Instance | Serverless |
| Libraries | None — installed via `%pip` below |

### Prerequisites
- `setup/01_prepare_test_cases` — TC PDFs must exist in Volume
- `vllm_batch/notebook` — must be configured as a Databricks Job
- Set the `job_id` widget to the batch job's ID before running

### Widgets
- **`job_id`** (required): The Databricks Job ID for the vLLM_Batch notebook

### Install Dependencies

In [ ]:
%pip install "databricks-sdk>=0.28" pymupdf pyyaml

### Configuration and SDK Setup

Initialises the Databricks SDK `WorkspaceClient` for job management (trigger, poll status). Uses the notebook's own API token for authentication.

In [ ]:
import yaml, os, base64, time, uuid
from datetime import datetime, timezone

# Resolve project root
if "__file__" in dir():
    _root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
else:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _root = "/Workspace" + os.path.dirname(os.path.dirname(_nb))

cfg           = yaml.safe_load(open(f"{_root}/config.yaml"))
CATALOG       = cfg["catalog"]
SCHEMA        = cfg["schema"]
QUEUE_TABLE   = f"{CATALOG}.{SCHEMA}.{cfg['queue_table']}"
PERF_TABLE    = f"{CATALOG}.{SCHEMA}.{cfg['perf_table']}"
RESULTS_TABLE = f"{CATALOG}.{SCHEMA}.{cfg['batch_results_table']}"
TC_PATH       = f"/Volumes/{CATALOG}/{SCHEMA}/{cfg['volume']}/{cfg['test_cases_subpath']}"

# Read job ID from notebook widget
dbutils.widgets.text("job_id", "")
JOB_ID        = int(dbutils.widgets.get("job_id"))
POLL_INTERVAL = 15     # seconds between queue table checks
JOB_TIMEOUT   = 3600   # max wall-clock time for the entire test run

from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F
import fitz  # PyMuPDF — used to count pages

# Authenticate SDK with this notebook's token
_host  = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
w = WorkspaceClient(host=_host, token=_token)

print(f"Job ID        : {JOB_ID}")
print(f"Queue table   : {QUEUE_TABLE}")
print(f"Results table : {RESULTS_TABLE}")
print(f"TC path       : {TC_PATH}")

### Ensure Tables Exist

Creates the queue table, performance results table, and parsed results table if they don't already exist.

In [ ]:
# Queue table — shared with vllm_batch/notebook.ipynb
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QUEUE_TABLE} (
    request_id  STRING    NOT NULL,
    pdf_base64  STRING,
    status      STRING,
    markdown    STRING,
    char_count  LONG,
    error       STRING,
    created_at  TIMESTAMP,
    updated_at  TIMESTAMP
) USING DELTA
""")
print(f"Queue table ready: {QUEUE_TABLE}")

# Performance results table — shared across all deployment options
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {PERF_TABLE} (
  option STRING, tc_id STRING, cold_start_s DOUBLE,
  model_load_s DOUBLE, latency_s DOUBLE, pages LONG, ts TIMESTAMP
) USING DELTA
""")
print(f"Perf table ready: {PERF_TABLE}")

# Parsed results table — stores extracted markdown per test case
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {RESULTS_TABLE} (
  tc_id STRING, pages LONG, markdown STRING, char_count LONG, ts TIMESTAMP
) USING DELTA
""")
print(f"Results table ready: {RESULTS_TABLE}")

### Helper Functions

- **`enqueue_pdf`**: Reads a PDF file, base64-encodes it, and inserts a `pending` row into the queue table. Returns the `request_id` and page count.
- **`_check_batch_job`**: Checks if the batch job run has failed or been cancelled (raises `RuntimeError` if so).
- **`poll_result`**: Polls the queue table until the request transitions to `done` or `error`, periodically checking the batch job for early failure detection.

In [ ]:
def enqueue_pdf(pdf_path: str) -> tuple:
    """Read PDF from Volume, base64-encode, insert as pending row. Returns (request_id, page_count)."""
    doc = fitz.open(pdf_path)
    pages = len(doc); doc.close()
    with open(pdf_path, "rb") as f:
        pdf_b64 = base64.b64encode(f.read()).decode()
    rid = str(uuid.uuid4())
    now = datetime.now(timezone.utc)
    spark.createDataFrame([{
        "request_id": rid, "pdf_base64": pdf_b64,
        "status": "pending", "markdown": "", "char_count": 0,
        "error": "", "created_at": now, "updated_at": now,
    }]).write.mode("append").saveAsTable(QUEUE_TABLE)
    return rid, pages

def _check_batch_job(batch_run_id: int) -> None:
    """Raise RuntimeError if the batch job run has failed or been cancelled."""
    r = w.jobs.get_run(run_id=batch_run_id)
    lc = r.state.life_cycle_state.value if r.state and r.state.life_cycle_state else ""
    rs = r.state.result_state.value if r.state and r.state.result_state else ""
    if lc in ("TERMINATED", "INTERNAL_ERROR", "SKIPPED"):
        if rs != "SUCCESS":
            msg = r.state.state_message if r.state else ""
            raise RuntimeError(f"Batch job failed: {lc}/{rs} — {msg}")

def poll_result(rid: str, deadline: float, batch_run_id: int = None) -> dict:
    """Poll queue table for result. Checks job health every 4th iteration."""
    check_counter = 0
    while time.time() < deadline:
        row = (spark.table(QUEUE_TABLE)
               .filter(F.col("request_id") == rid)
               .select("status", "markdown", "char_count", "error")
               .first())
        if row and row.status in ("done", "error"):
            return row.asDict()
        # Periodically verify the batch job hasn't crashed
        check_counter += 1
        if batch_run_id and check_counter % 4 == 0:
            _check_batch_job(batch_run_id)
        time.sleep(POLL_INTERVAL)
    return {"status": "timeout", "markdown": "", "char_count": 0, "error": "timed out"}

### Enqueue Test PDFs

Inserts TC1–TC4 into the queue table as `pending` rows. The batch job will pick these up when triggered.

In [ ]:
tc_files    = {"TC1": "tc1.pdf", "TC2": "tc2.pdf", "TC3": "tc3.pdf", "TC4": "tc4.pdf"}
request_ids = {}   # tc_id → request_id mapping
page_counts = {}   # tc_id → number of pages

print("Enqueuing test PDFs ...")
for tc_id, fname in tc_files.items():
    rid, pages         = enqueue_pdf(os.path.join(TC_PATH, fname))
    request_ids[tc_id] = rid
    page_counts[tc_id] = pages
    print(f"  {tc_id} → {rid}")

### Trigger Batch Job

Starts a new run of the vLLM_Batch job. The timer starts here — `cold_start_s` measures from trigger to first result (includes cluster startup + model loading + inference).

In [ ]:
print(f"\nTriggering job {JOB_ID} ...")
t_trigger = time.time()  # Start wall-clock timer (includes cold start)
run       = w.jobs.run_now(job_id=JOB_ID)
print(f"  Run ID  : {run.run_id}")
print(f"  Monitor : {w.config.host}/jobs/{JOB_ID}/runs/{run.run_id}")

### Poll for Results

Waits for each test case to transition from `pending` → `done` in the queue table. A test case **passes** if it returns more than 5 characters of markdown.

Note: `latency_s` here equals `cold_start_s` because the timer starts at job trigger (not model readiness). This is by design — batch jobs include cold start in their total latency.

In [ ]:
print(f"\nPolling for results (timeout={JOB_TIMEOUT}s) ...")
deadline     = time.time() + JOB_TIMEOUT
batch_run_id = run.run_id
results      = {}    # tc_id → "PASS" or "FAIL: reason"
records      = []    # perf rows to write
parsed       = []    # parsed markdown rows to write

for tc_id, rid in request_ids.items():
    result  = poll_result(rid, deadline, batch_run_id=batch_run_id)
    latency = time.time() - t_trigger  # Wall clock from trigger

    if result["status"] == "done":
        ok = result["char_count"] > 5
        results[tc_id] = "PASS" if ok else f"FAIL: {result['char_count']} chars"
        print(f"  {tc_id}: {results[tc_id]} ({result['char_count']} chars, {latency:.1f}s)")
        records.append({
            "option": "vLLM_Batch", "tc_id": tc_id, "cold_start_s": latency,
            "model_load_s": 0.0, "latency_s": latency,
            "pages": page_counts[tc_id], "ts": datetime.now(timezone.utc),
        })
        parsed.append({
            "tc_id": tc_id, "pages": page_counts[tc_id],
            "markdown": result["markdown"], "char_count": result["char_count"],
            "ts": datetime.now(timezone.utc),
        })
    else:
        results[tc_id] = f"FAIL: {result['status']} — {result['error']}"
        print(f"  {tc_id}: {results[tc_id]}")

### Write Results and Summary

Appends parsed markdown to the results table, latency records to the perf table, and prints a pass/fail summary.

In [ ]:
# Write parsed results to the results table
if parsed:
    spark.createDataFrame(parsed).write.mode("append").saveAsTable(RESULTS_TABLE)
    print(f"Wrote {len(parsed)} parsed result(s) to {RESULTS_TABLE}")

# Write perf records to the shared results table
if records:
    spark.createDataFrame(records).write.mode("append").saveAsTable(PERF_TABLE)
    print(f"Wrote {len(records)} perf record(s) to {PERF_TABLE}")

# Print summary banner
print("\n" + "=" * 60)
print("TEST SUMMARY — vLLM_Batch")
print("=" * 60)
for tc, res in results.items():
    print(f"  [{'PASS' if res == 'PASS' else 'FAIL'}] {tc}: {res}")
passed = sum(1 for v in results.values() if v == "PASS")
print(f"\n{passed}/{len(results)} tests passed")

# Surface results via Jobs API (serverless stdout is not captured)
import json as _json
dbutils.notebook.exit(_json.dumps({"passed": passed, "total": len(results), "results": results}))